# Exercise 02: AlphaFold Structure Prediction and Rational Protein Design

## Learning Objectives

In this exercise, you will:
- Use AlphaFold to predict protein structures
- **Design sequences** that fold into specific secondary structures
- **Interpret confidence metrics** (pLDDT, PAE) critically
- **Hypothesize and test** structure-disrupting mutations
- **Compare** predicted vs experimental structures
- **Justify** your design choices using structural biology principles

## About AI and This Exercise

**What AI CAN help with:**
- Understanding AlphaFold parameters
- Syntax for running predictions
- General protein folding principles

**What YOU must demonstrate:**
- **Design strategy** - WHY you chose specific amino acids
- **Hypothesis formulation** - predictions before running AlphaFold
- **Critical evaluation** - interpreting what worked and what didn't
- **Biological reasoning** - connecting sequence to structure to function

**Important:** Simply generating sequences with AI and testing them randomly shows no understanding. You must explain your reasoning.

---

Based on [Sergey Ovchinnikov's af_backprop](https://github.com/sokrypton/af_backprop)

# Alphafold

[AlphaFold](https://deepmind.google/technologies/alphafold/), an AI system developed by DeepMind, has solved the complex protein-folding problem, allowing for almost instant and highly accurate predictions of protein structures, which are crucial for understanding cellular functions and advancing medical research. Recognized by the [Critical Assessment of protein Structure Prediction community](https://www.predictioncenter.org/), AlphaFold has [significantly](https://www.predictioncenter.org/casp14/zscores_final.cgi) expanded the availability of protein structure data through the freely accessible AlphaFold Protein Structure Database.

In [1]:
# @title Setup Cell 🏗️
# @markdown This cell sets up the complete AlphaFold environment including downloading parameters, installing dependencies, and defining prediction functions.
# @markdown **Run this cell first before making any predictions.**

# ============================================
# JAX VERSION FIX FOR GOOGLE COLAB
# ============================================
# Check if running in Colab and install compatible JAX version
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🔧 Installing JAX compatible with AlphaFold...")
    import subprocess

    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "jax[cuda12]==0.5.3",
            "jaxlib==0.5.3",
            "-q",
        ],
        capture_output=True,
    )
    if result.returncode == 0:
        print("✅ JAX 0.5.3 installed successfully")
        print(
            "⚠️  IMPORTANT: Please restart the runtime now (Runtime → Restart runtime)"
        )
        print("    Then run this cell again to continue with the setup.")
    else:
        print("❌ Failed to install JAX. Error:", result.stderr.decode())
else:
    print("ℹ️  Not in Colab, skipping JAX version check")

print("\n" + "=" * 60)
print("Setting up AlphaFold environment...")
print("=" * 60 + "\n")

import logging
import os
import re
import warnings
import hashlib
from IPython.display import display


import logging

logger = logging.getLogger("ex02")
logging.basicConfig(level=logging.INFO)
logger.setLevel(logging.INFO)

# Suppress various warnings
# warnings.simplefilter(action="ignore", category=FutureWarning)
# warnings.simplefilter(action="ignore", category=SyntaxWarning)

# Suppress TensorFlow warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppress all TF messages except errors
os.environ["XLA_FLAGS"] = "--xla_hlo_profile"  # Suppress XLA warnings
os.environ["JAX_PLATFORMS"] = "cpu"  # Force JAX to use CPU to avoid CUDA warnings

# Suppress CUDA and compilation warnings
logging.getLogger("absl").setLevel(logging.ERROR)  # Suppress absl/MLIR messages

# Additional JAX/XLA suppression
os.environ["JAX_LOG_COMPILES"] = "0"  # Suppress JAX compilation logs
os.environ["XLA_FLAGS"] = "--xla_hlo_profile=false"  # Suppress XLA profiling messages


# Configuration for parameter download
EXPECTED_MD5 = "7604d0da26bca1c36d64d7bb89ba7751"
EXPECTED_SIZE = 3722752000  # ~3.5GB
TAR_FILE = "alphafold_params_2021-07-14.tar"
PARAMS_DIR = "params"
DOWNLOAD_URL = (
    "https://storage.googleapis.com/alphafold/alphafold_params_2021-07-14.tar"
)


def calculate_md5(filename, chunk_size=8192):
    """Calculate MD5 hash of a file.

    Args:
        filename (str): Path to the file to calculate MD5 for.
        chunk_size (int, optional): Size of chunks to read at a time in bytes.
            Defaults to 8192.

    Returns:
        str or None: MD5 hash as hexadecimal string, or None if file not found.
    """
    hash_md5 = hashlib.md5()
    try:
        with open(filename, "rb") as f:
            for chunk in iter(lambda: f.read(chunk_size), b""):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()
    except FileNotFoundError:
        return None


def download_with_progress(url, filename):
    """Download file from URL with progress bar using requests and tqdm.

    Args:
        url (str): URL to download from.
        filename (str): Local filename to save the downloaded file.

    Raises:
        requests.RequestException: If download fails.
    """
    import requests
    from tqdm import tqdm

    print(f"🐍 Downloading with Python + tqdm...")
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get("content-length", 0))

    with (
        open(filename, "wb") as file,
        tqdm(
            desc=filename,
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as progress_bar,
    ):
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                file.write(chunk)
                progress_bar.update(len(chunk))


if "SETUP_DONE" not in dir():
    logger.info("🚧 Setup environment")
    from IPython.utils import io
    from IPython.display import HTML

    # Import TensorFlow and JAX with suppressed warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        import tensorflow as tf
        import jax
        import jax.numpy as jnp

    import numpy as np
    import matplotlib
    from matplotlib import animation
    import matplotlib.pyplot as plt
    import tqdm.notebook

    TQDM_BAR_FORMAT = (
        "{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]"
    )

    with io.capture_output() as captured:
        # Clone af_backprop if not exists
        if not os.path.isdir("af_backprop"):
            logger.info("🧩 Downloading and installing af_backprop")
            !git clone https://github.com/yerkoescalona/af_backprop.git
            !pip -q install biopython dm-haiku ml-collections py3Dmol
            !wget -qnc https://raw.githubusercontent.com/sokrypton/ColabFold/main/beta/colabfold.py

        # Download AlphaFold parameters if not exists
        if not os.path.isdir(PARAMS_DIR):
            logger.info("🧩 Downloading AlphaFold parameters")
            os.makedirs(PARAMS_DIR, exist_ok=True)

            need_download = True

            # Check if tar file already exists and verify integrity
            if os.path.exists(TAR_FILE):
                logger.info(f"📁 Found existing tar file: {TAR_FILE}")
                file_size = os.path.getsize(TAR_FILE)
                logger.info(f"📏 File size: {file_size / (1024 * 1024 * 1024):.2f} GB")

                if file_size == EXPECTED_SIZE:
                    logger.info("🔍 Verifying file integrity with MD5 checksum...")
                    file_md5 = calculate_md5(TAR_FILE)
                    if file_md5 == EXPECTED_MD5:
                        logger.info("✅ File integrity verified! Skipping download.")
                        need_download = False
                    else:
                        logger.warning(
                            f"❌ MD5 mismatch! Expected: {EXPECTED_MD5}, Got: {file_md5}"
                        )
                        logger.info("🗑️ Removing corrupted file and will re-download...")
                        os.remove(TAR_FILE)
                else:
                    logger.warning(
                        f"⚠️ File size mismatch! Expected: {EXPECTED_SIZE / (1024**3):.2f} GB, Got: {file_size / (1024**3):.2f} GB"
                    )
                    logger.info("🗑️ Removing incomplete file and will re-download...")
                    os.remove(TAR_FILE)

            if need_download:
                logger.info("⬇️ Starting download of AlphaFold parameters (~3.5GB)")

                # Install aria2c for faster downloads
                !apt-get install aria2 -qq

                # Try aria2c first (fastest, multi-threaded)
                logger.info(
                    "🚀 Attempting fast download with aria2c (16 connections)..."
                )
                !aria2c -q -x 16 --file-allocation=none --check-certificate=false -o {TAR_FILE} {DOWNLOAD_URL}

                # Check if aria2c download succeeded
                if (
                    not os.path.exists(TAR_FILE)
                    or os.path.getsize(TAR_FILE) < 1000000000
                ):
                    logger.warning("❌ aria2c failed or unavailable")
                    if os.path.exists(TAR_FILE):
                        os.remove(TAR_FILE)

                    # Try wget next
                    logger.info("🔄 Trying wget...")
                    !wget --progress=bar:force -O {TAR_FILE} {DOWNLOAD_URL}

                    # Check if wget succeeded
                    if (
                        not os.path.exists(TAR_FILE)
                        or os.path.getsize(TAR_FILE) < 1000000000
                    ):
                        logger.warning("❌ wget failed or unavailable")
                        if os.path.exists(TAR_FILE):
                            os.remove(TAR_FILE)

                        # Fallback to Python with tqdm
                        try:
                            download_with_progress(DOWNLOAD_URL, TAR_FILE)
                        except Exception as e:
                            logger.error(f"❌ All download methods failed: {str(e)}")
                            raise

                # Verify downloaded file
                if os.path.exists(TAR_FILE):
                    file_size = os.path.getsize(TAR_FILE)
                    logger.info(
                        f"✅ Downloaded file size: {file_size / (1024 * 1024 * 1024):.2f} GB"
                    )

                    logger.info("🔍 Verifying file integrity with MD5 checksum...")
                    file_md5 = calculate_md5(TAR_FILE)
                    if file_md5 == EXPECTED_MD5:
                        logger.info("✅ File integrity verified! MD5 checksum matches.")
                    else:
                        logger.error(
                            f"❌ MD5 mismatch! Expected: {EXPECTED_MD5}, Got: {file_md5}"
                        )
                        logger.error(
                            "⚠️ Downloaded file may be corrupted. Please delete the file and try again."
                        )
                        raise RuntimeError("File integrity check failed")
                else:
                    logger.error("❌ Download failed: file not found")
                    raise FileNotFoundError(f"Could not download {TAR_FILE}")

            # Extract parameters
            logger.info("📦 Extracting AlphaFold parameters...")
            import tarfile

            with tarfile.open(TAR_FILE) as tar:
                tar.extractall(path=PARAMS_DIR)
            logger.info("✅ Parameters extracted successfully!")

    # Configure which device to use
    try:
        # Check if TPU is available
        import jax.tools.colab_tpu

        jax.tools.colab_tpu.setup_tpu()
        logger.info("Running on TPU")
        DEVICE = "tpu"
        logger.critical("☠️ TPU is not supported 13.11.24")
        raise NotImplementedError("TPU is not supported")
    except:
        if jax.local_devices()[0].platform == "cpu":
            logger.warning("WARNING: no GPU detected, will be using CPU")
            DEVICE = "cpu"
        else:
            logger.info("Running on GPU")
            DEVICE = "gpu"
            # Disable GPU on tensorflow
            tf.config.set_visible_devices([], "GPU")

    # Import libraries
    sys.path.append("af_backprop")

    SETUP_DONE = True

logger.info("✅ Setup done")

if "LIBRARY_IMPORTED" not in dir():
    logger.info("🚧 Import libraries")
    from utils import update_seq, update_aatype, get_plddt, get_pae
    import colabfold as cf
    from alphafold.common import protein
    from alphafold.data import pipeline
    from alphafold.model import data, config, model
    from alphafold.common import residue_constants

    # Custom functions
    def clear_mem():
        """Clear GPU/TPU memory by deleting all live buffers."""
        try:
            # JAX 0.4.x compatible API
            backend = jax.lib.xla_bridge.get_backend()
            for buf in backend.live_buffers():
                buf.delete()
        except AttributeError:
            # Fallback for other JAX versions
            import gc

            gc.collect()

    def setup_model(max_len, model_name="model_3_ptm"):
        clear_mem()

        # Setup model
        cfg = config.model_config("model_5_ptm")
        cfg.model.num_recycle = 0
        cfg.data.common.num_recycle = 0
        cfg.data.eval.max_msa_clusters = 1
        cfg.data.common.max_extra_msa = 1
        cfg.data.eval.masked_msa_replace_fraction = 0
        cfg.model.global_config.subbatch_size = None
        model_params = data.get_model_haiku_params(model_name=model_name, data_dir=".")
        model_runner = model.RunModel(cfg, model_params, is_training=False)

        seq = "A" * max_len
        length = len(seq)
        feature_dict = {
            **pipeline.make_sequence_features(
                sequence=seq, description="none", num_res=length
            ),
            **pipeline.make_msa_features(
                msas=[[seq]], deletion_matrices=[[[0] * length]]
            ),
        }
        inputs = model_runner.process_features(feature_dict, random_seed=0)

        def runner(I):
            # Update sequence
            inputs = I["inputs"]
            inputs["prev"] = I["prev"]

            seq = jax.nn.one_hot(I["seq"], 20)
            update_seq(seq, inputs)
            update_aatype(inputs["target_feat"][..., 1:], inputs)

            # Mask prediction
            mask = jnp.arange(inputs["residue_index"].shape[0]) < I["length"]
            inputs["seq_mask"] = inputs["seq_mask"].at[:].set(mask)
            inputs["msa_mask"] = inputs["msa_mask"].at[:].set(mask)
            inputs["residue_index"] = jnp.where(mask, inputs["residue_index"], 0)

            # Get prediction
            key = jax.random.PRNGKey(0)
            outputs = model_runner.apply(I["params"], key, inputs)

            aux = {
                "final_atom_positions": outputs["structure_module"][
                    "final_atom_positions"
                ],
                "final_atom_mask": outputs["structure_module"]["final_atom_mask"],
                "plddt": get_plddt(outputs),
                "pae": get_pae(outputs),
                "length": I["length"],
                "seq": I["seq"],
                "prev": outputs["prev"],
                "residue_idx": inputs["residue_index"][0],
            }
            return aux

        return jax.jit(runner), {
            "inputs": inputs,
            "params": model_params,
            "length": max_len,
        }

    def save_pdb(outs, filename):
        """Save pdb coordinates"""
        p = {
            "residue_index": outs["residue_idx"] + 1,
            "aatype": outs["seq"],
            "atom_positions": outs["final_atom_positions"],
            "atom_mask": outs["final_atom_mask"],
            "plddt": outs["plddt"],
        }
        p = jax.tree_util.tree_map(lambda x: x[: outs["length"]], p)
        b_factors = 100 * p.pop("plddt")[:, None] * p["atom_mask"]
        p = protein.Protein(**p, b_factors=b_factors)
        pdb_lines = protein.to_pdb(p)
        with open(filename, "w") as f:
            f.write(pdb_lines)

    def make_animation(positions, plddts, Ls=None, line_w=2.0, dpi=100):
        def ca_align_to_last(positions):
            def align(P, Q):
                p = P - P.mean(0, keepdims=True)
                q = Q - Q.mean(0, keepdims=True)
                return p @ cf.kabsch(p, q)

            pos = positions[-1, :, 1, :] - positions[-1, :, 1, :].mean(0, keepdims=True)
            best_2D_view = pos @ cf.kabsch(pos, pos, return_v=True)

            new_positions = []
            for i in range(len(positions)):
                new_positions.append(align(positions[i, :, 1, :], best_2D_view))
            return np.asarray(new_positions)

        # Align all to last recycle
        pos = ca_align_to_last(positions)

        fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
        fig.subplots_adjust(top=0.90, bottom=0.10, right=1, left=0, hspace=0, wspace=0)
        fig.set_figwidth(13)
        fig.set_figheight(5)
        fig.set_dpi(dpi)

        xy_min = pos[..., :2].min() - 1
        xy_max = pos[..., :2].max() + 1

        for ax in [ax1, ax3]:
            ax.set_xlim(xy_min, xy_max)
            ax.set_ylim(xy_min, xy_max)
            ax.axis(False)

        ax2.set_xlabel("positions")
        ax2.set_ylabel("pLDDT")
        ax2.set_ylim(0, 100)

        ims = []
        for k, (xyz, plddt) in enumerate(zip(pos, plddts)):
            ims.append([])
            im2 = ax2.plot(plddt, animated=True, color="black")[0]
            tt2 = cf.add_text(f"recycle={k}", ax2)
            tt3 = cf.add_text(f"pLDDT={plddt.mean():.3f}", ax3)
            if Ls is None or len(Ls) == 1:
                tt1 = cf.add_text("colored by N->C", ax1)
                ims[-1] += [cf.plot_pseudo_3D(xyz, ax=ax1, line_w=line_w)]
            else:
                # Color by chain
                tt1 = cf.add_text("colored by chain", ax1)
                c = np.concatenate([[n] * L for n, L in enumerate(Ls)])
                ims[-1] += [
                    cf.plot_pseudo_3D(
                        xyz,
                        c=c,
                        cmap=cf.pymol_cmap,
                        cmin=0,
                        cmax=39,
                        line_w=line_w,
                        ax=ax1,
                    )
                ]

            ims[-1] += [im2, tt1, tt2, tt3]
            ims[-1] += [
                cf.plot_pseudo_3D(xyz, c=plddt, cmin=50, cmax=90, ax=ax3, line_w=line_w)
            ]

        ani = animation.ArtistAnimation(fig, ims, blit=True, interval=120)
        plt.close()
        return ani.to_html5_video()

    LIBRARY_IMPORTED = True

logger.info("✅ Libraries imported")


# Define the main prediction function
def predict_structure(
    sequence,
    recycles=48,
    color="confidence",
    show_sidechains=True,
    show_mainchains=False,
):
    """Predict protein structure using AlphaFold and display interactive results.

    This function uses AlphaFold neural network to predict the 3D structure of a
    protein from its amino acid sequence. It displays the predicted structure using
    py3Dmol and generates confidence plots using matplotlib.

    Args:
        sequence (str): Amino acid sequence using single letter codes (A-Z).
            Use "/" to specify chain breaks (e.g., "AAA/AAA" for two chains).
        recycles (int, optional): Number of recycles to perform during prediction.
            More recycles generally improve accuracy but take longer. Defaults to 48.
        color (str, optional): Coloring scheme for 3D visualization. Options are:
            - "confidence": Color by confidence scores (lDDT)
            - "rainbow": Color from N-terminus (blue) to C-terminus (red)
            - "chain": Color by chain ID for multi-chain proteins
            Defaults to "confidence".
        show_sidechains (bool, optional): Whether to display side chains in 3D
            structure. Defaults to True.
        show_mainchains (bool, optional): Whether to display backbone bonds in 3D
            structure. Defaults to False.

    Returns:
        None: Function displays interactive visualizations directly in notebook.

    Raises:
        RuntimeError: If sequence is empty or contains invalid characters.

    Note:
        The function maintains internal state to avoid recompilation when predicting
        similar length sequences. First prediction may take longer due to model
        compilation.

    Example:
        >>> predict_structure("MKQHKAMIVALIVICITAVVAAL")  # Single chain
        >>> predict_structure("AAA/BBB", recycles=24)      # Two chains
        >>> predict_structure("SEQUENCE", color="rainbow") # Rainbow coloring
    """
    # Initialize
    if "current_seq" not in globals():
        global current_seq, r, max_length, runner, I, outs, positions, plddts, paes
        current_seq = ""
        r = -1
        max_length = -1

    # Collect user inputs
    ori_sequence = re.sub("[^A-Z/:]", "", sequence.upper())

    # Check if the sequence is empty after cleaning
    if not ori_sequence:
        logger.critical(
            "❌ Input sequence is empty or invalid. Please enter a valid amino acid sequence.".upper()
        )
        return None

    Ls = [len(s) for s in ori_sequence.replace(":", "/").split("/")]
    sequence = re.sub("[^A-Z]", "", ori_sequence)
    length = len(sequence)

    # Avoid recompiling if length within 25
    if length > max_length or (max_length - length) > 25:
        max_length = length + 25
        print("🔧 Compiling AlphaFold model (this may take a moment)...")

        # Suppress compilation warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            runner, I = setup_model(max_length)

        print("✅ Model compiled successfully!")

    if ori_sequence != current_seq:
        outs = []
        positions = []
        plddts = []
        paes = []
        r = -1

        # Pad sequence to max length
        seq = np.array([residue_constants.restype_order.get(aa, 0) for aa in sequence])
        seq = np.pad(seq, [0, max_length - length], constant_values=-1)

        # Update inputs, restart recycle
        I.update(
            {
                "seq": seq,
                "length": length,
                "prev": {
                    "prev_msa_first_row": np.zeros([max_length, 256]),
                    "prev_pair": np.zeros([max_length, max_length, 128]),
                    "prev_pos": np.zeros([max_length, 37, 3]),
                },
            }
        )

        I["inputs"]["residue_index"][:] = cf.chain_break(
            np.arange(max_length), Ls, length=32
        )
        current_seq = ori_sequence

    # Run for defined number of recycles
    with tqdm.notebook.tqdm(total=(recycles + 1), bar_format=TQDM_BAR_FORMAT) as pbar:
        p = 0
        while p < min(r + 1, recycles + 1):
            pbar.update(1)
            p += 1
        while r < recycles:
            O = runner(I)
            O = jax.tree_util.tree_map(lambda x: np.asarray(x), O)
            positions.append(O["final_atom_positions"][:length])
            plddts.append(O["plddt"][:length])
            paes.append(O["pae"][:length, :length])
            I["prev"] = O["prev"]
            outs.append(O)
            r += 1
            pbar.update(1)

    if color == "confidence":
        color = "lDDT"

    print(f"plotting prediction at recycle={recycles}")
    save_pdb(outs[recycles], "out.pdb")
    v = cf.show_pdb(
        "out.pdb",
        show_sidechains,
        show_mainchains,
        color,
        color_HP=True,
        size=(800, 480),
        Ls=Ls,
    )
    v.setHoverable(
        {},
        True,
        """function(atom,viewer,event,container){if(!atom.label){atom.label=viewer.addLabel("      "+atom.resn+":"+atom.resi,{position:atom,backgroundColor:'mintcream',fontColor:'black'});}}""",
        """function(atom,viewer){if(atom.label){viewer.removeLabel(atom.label);delete atom.label;}}""",
    )
    v.show()
    if color == "lDDT":
        # Show matplotlib version of pLDDT legend
        cf.plot_plddt_legend().show()

    # Add confidence plots (matplotlib version)
    cf.plot_confidence(plddts[recycles] * 100, paes[recycles], Ls=Ls).show()


logger.info("✅ predict_structure function defined and ready to use!")
print("=" * 80)
print("🎉 Setup Complete! You can now run predictions using predict_structure()")
print("=" * 80)

INFO:ex02:🚧 Setup environment


ℹ️  Not in Colab, skipping JAX version check

Setting up AlphaFold environment...



I0000 00:00:1786198904.593016   23075 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786198924.331153   23075 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
INFO:ex02:🧩 Downloading and installing af_backprop
INFO:ex02:🧩 Downloading AlphaFold parameters
INFO:ex02:⬇️ Starting download of AlphaFold parameters (~3.5GB)
INFO:ex02:🚀 Attempting fast download with aria2c (16 connections)...
INFO:ex02:✅ Downloaded file size: 3.47 GB
INFO:ex02:🔍 Verifying file integrity with MD5 checksum...
INFO:ex02:✅ File integrity verified! MD5 checksum matches.
INFO:ex02:📦 Extracting AlphaFold parameters...
INFO:ex02:✅ Parameters extracted successfully!
INFO:ex02:✅ Setup done
INFO:ex02:🚧 Import libraries
INFO:ex02:✅ Libraries imported
INFO:ex02:✅ predict_structure function defined and ready to use!


🎉 Setup Complete! You can now run predictions using predict_structure()


## Alphafold_single Tasks

This tutorial is based on [Alphafold_single](https://colab.research.google.com/github/sokrypton/af_backprop/blob/beta/examples/AlphaFold_single.ipynb). Follow the instructions:

- Patience... The first time you run the cell below it will take 1 minitue to setup, after that it should run in seconds (after each change).
- click the little ▶ play icon to the left of each cell below.
- For 3D display, hold mouseover aminoacid to get name and position number
- use "/" to specify chainbreaks, (eg. sequence="AAA/AAA")

Tasks:

 1. Change the sequence of aminoacids to a different one (e.g. all Alanines, all Leucines, etc.). What is the difference in the predicted structure?

 1. Design a sequence that fold into a Helix (see figure)

 2. Design a sequence that fold into Two Helices (see figure)

 3. Design a sequence that fold into Beta Sheets (see figure)

<img src="https://github.com/yerkoescalona/structural_bioinformatics/blob/w2023/ex02/challenge1.png?raw=1" alt="Drawing"/>

 5. Design a sequence that fold into 4 helix bundle (take a look at e.g. 3VJF)





 **Hint**: Use the following information to design your sequences

<img src="https://github.com/yerkoescalona/structural_bioinformatics/blob/w2023/ex02/statistics.png?raw=1" alt="Drawing"/>



**Note on this mode:** `alphafold_single` predicts from the query sequence **alone** — no multiple sequence alignment (MSA), no templates, no evolutionary information at all. Keep that in mind; further down in this notebook we'll contrast it with a real, full AlphaFold DB prediction (which *does* use a deep MSA) for an actual, well-studied protein.


In [2]:
# Cell 3: Predict Structure for a Sequence

# @title Change the sequence of aminoacids to a different one (e.g. all Alanines, all Leucines, etc.). What is the difference in the predicted structure?
sequence = "GGGGGGGGGGG"  # @param {type:"string"}
recycles = 48  # @param ["0", "1", "2", "3", "6", "12", "24", "48"] {type:"raw"}
color = "confidence"  # @param ["chain", "confidence", "rainbow"]
show_sidechains = True  # @param {type:"boolean"}
show_mainchains = True  # @param {type:"boolean"}

predict_structure(
    sequence=sequence,
    recycles=recycles,
    color=color,
    show_sidechains=show_sidechains,
    show_mainchains=show_mainchains,
)


🔧 Compiling AlphaFold model (this may take a moment)...
✅ Model compiled successfully!


  0%|          | 0/49 [elapsed: 00:00 remaining: ?]

AttributeError: module 'jax' has no attribute 'tree_map'

In [ ]:
# Cell 3: Predict Structure for a Sequence

# @title Design sequences that fold into a ALPHA HELIX secondary structures (see figure above)
sequence = ""  # @param {type:"string"}
recycles = 48  # @param ["0", "1", "2", "3", "6", "12", "24", "48"] {type:"raw"}
color = "rainbow"  # @param ["chain", "confidence", "rainbow"]
show_sidechains = True  # @param {type:"boolean"}
show_mainchains = True  # @param {type:"boolean"}

predict_structure(
    sequence,
    recycles=recycles,
    color=color,
    show_sidechains=show_sidechains,
    show_mainchains=show_mainchains,
)


In [ ]:
# Cell 3: Predict Structure for a Sequence

# @title Design sequences that fold into TWO HELICES (see figure above)
sequence = ""  # @param {type:"string"}
recycles = 48  # @param ["0", "1", "2", "3", "6", "12", "24", "48"] {type:"raw"}
color = "rainbow"  # @param ["chain", "confidence", "rainbow"]
show_sidechains = True  # @param {type:"boolean"}
show_mainchains = True  # @param {type:"boolean"}

predict_structure(
    sequence,
    recycles=recycles,
    color=color,
    show_sidechains=show_sidechains,
    show_mainchains=show_mainchains,
)


In [ ]:
# Cell 3: Predict Structure for a Sequence

# @title Design sequences that fold into BETA SHEETS (see figure above)
sequence = ""  # @param {type:"string"}
recycles = 48  # @param ["0", "1", "2", "3", "6", "12", "24", "48"] {type:"raw"}
color = "rainbow"  # @param ["chain", "confidence", "rainbow"]
show_sidechains = True  # @param {type:"boolean"}
show_mainchains = True  # @param {type:"boolean"}

predict_structure(
    sequence,
    recycles=recycles,
    color=color,
    show_sidechains=show_sidechains,
    show_mainchains=show_mainchains,
)


In [ ]:
# Cell 3: Predict Structure for a Sequence

# @title Design a sequence that fold into 4 helix bundle (take a look at e.g. 3VJF)
sequence = ""  # @param {type:"string"}
recycles = 48  # @param ["0", "1", "2", "3", "6", "12", "24", "48"] {type:"raw"}
color = "rainbow"  # @param ["chain", "confidence", "rainbow"]
show_sidechains = True  # @param {type:"boolean"}
show_mainchains = True  # @param {type:"boolean"}

predict_structure(
    sequence,
    recycles=recycles,
    color=color,
    show_sidechains=show_sidechains,
    show_mainchains=show_mainchains,
)


---

## Reading AlphaFold's confidence metrics: a real worked example

The tasks above used `alphafold_single` in its stripped-down, single-sequence mode —
good for exploring how AlphaFold turns a sequence into a shape, but not representative
of how AlphaFold is normally used, or of how confident it can be on a real protein.

For that, we'll look at a real, extensively studied protein: the **core (DNA-binding)
domain of human p53** — "the guardian of the genome", the most commonly mutated gene
in human cancer. Specifically, PDB entry **3D08** carries a real cancer-hotspot
mutation, **R249S**, associated with hepatocellular carcinoma, and coordinates a
structural Zn²⁺ ion required for the domain to fold correctly.

We will not re-run AlphaFold ourselves here (that needs the full pipeline with a real
MSA, which is a GPU-hour-scale job) — instead we'll query the **AlphaFold Protein
Structure Database** directly, the same public, curated resource you'll use for your own
project protein in the assignment below. Below we'll go beyond the summary numbers:
real per-residue **pLDDT and PAE plots**, the **real MSA AlphaFold itself used** (yes,
AlphaFold DB tells you exactly how many sequences went into each prediction), and,
further down, a direct structural comparison against the 3D08 crystal itself.

**🎯 Predict first (calibration-graded, not correctness-graded):** Human p53
(UniProt `P04637`) is 393 residues long, but 3D08's crystal only resolved a domain in
the *middle* of that sequence (residues 97-287) — the rest wasn't modeled at all. Before
running the cells below: do you predict AlphaFold's confidence (pLDDT) will be roughly
**uniform** across all 393 residues, or **split** into confident and unconfident
stretches? If you predict a split, which part(s) of the sequence do you guess will be
the *confident* ones, and why?

*Your prediction:*


In [ ]:
# Query the AlphaFold DB REST API for the real, full-length human p53 prediction.
# This is a lightweight HTTP call -- no GPU, no jax, no af_backprop required.
import requests

uniprot_id = "P04637"  # human p53
afdb_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
afdb_entries = requests.get(afdb_url).json()

# The API can return more than one fragment for very long proteins; p53 fits in one.
entry = next(e for e in afdb_entries if e["modelEntityId"] == "AF-P04637-F1")

print(f"Model: {entry['modelEntityId']}  (pipeline: {entry['toolUsed']})")
print(f"Sequence length: {entry['sequenceEnd'] - entry['sequenceStart'] + 1} residues")
print(f"Global mean pLDDT: {entry['globalMetricValue']:.1f}")
print("Confidence-bin fractions (AlphaFold DB's own bins):")
print(f"  Very low (pLDDT < 50):  {entry['fractionPlddtVeryLow']*100:.1f}%")
print(f"  Low      (50-70):       {entry['fractionPlddtLow']*100:.1f}%")
print(f"  Confident(70-90):       {entry['fractionPlddtConfident']*100:.1f}%")
print(f"  Very high(>90):         {entry['fractionPlddtVeryHigh']*100:.1f}%")


Before interpreting that number, let's zoom in on exactly the region 3D08's crystal
actually covers (residues 97-287) — not the whole 393-residue protein.


In [ ]:
# Per-residue confidence for the same entry, restricted to 3D08's own resolved range.
# AlphaFold DB's own listing gives us the direct confidence-file URL for this entry.
confidence_json_url = entry["plddtDocUrl"]
conf = requests.get(confidence_json_url).json()

import pandas as pd

conf_df = pd.DataFrame({
    "residueNumber": conf["residueNumber"],
    "pLDDT": conf["confidenceScore"],
})

crystal_start, crystal_end = 97, 287  # 3D08's own resolved range
core_domain = conf_df[
    (conf_df["residueNumber"] >= crystal_start) & (conf_df["residueNumber"] <= crystal_end)
]
print(f"AlphaFold mean pLDDT over 3D08's resolved range ({crystal_start}-{crystal_end}): "
      f"{core_domain['pLDDT'].mean():.1f}")
print(f"Fraction of that range with pLDDT < 70: {(core_domain['pLDDT'] < 70).mean()*100:.1f}%")

r249_window = conf_df[(conf_df["residueNumber"] >= 240) & (conf_df["residueNumber"] <= 260)]
print("\npLDDT right around the R249S hotspot position (240-260):")
print(r249_window.to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt

# Plot pLDDT across the full 393-residue sequence, with 3D08's own resolved
# range (97-287) shaded, so the confident/unconfident split is visible directly
# instead of only read off printed numbers.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(conf_df["residueNumber"], conf_df["pLDDT"], color="tab:blue", linewidth=1)
ax.axvspan(crystal_start, crystal_end, color="tab:green", alpha=0.15,
           label=f"3D08 resolved range ({crystal_start}-{crystal_end})")
ax.axhline(70, color="grey", linestyle="--", linewidth=1, label="pLDDT = 70 (confident cutoff)")
ax.set_xlabel("Residue number (full-length p53, 1-393)")
ax.set_ylabel("pLDDT")
ax.set_title("AlphaFold DB per-residue confidence for human p53 (P04637)")
ax.legend(loc="lower center")
ax.set_ylim(0, 100)
plt.show()


### Predicted Aligned Error (PAE)

pLDDT tells you how confident AlphaFold is about *where a residue sits locally*. It does
not tell you how confident AlphaFold is about the **relative position of two residues
that are far apart in sequence** — two domains can each individually have high pLDDT
while AlphaFold is still unsure how they're oriented relative to each other. That's what
**PAE** (Predicted Aligned Error) is for: a full residue-by-residue matrix, where a low
value at (i, j) means AlphaFold is confident about residue j's position *if you fix
residue i in place*. AlphaFold DB publishes the full PAE matrix for every entry, the
same way it publishes pLDDT.

In [ ]:
# Fetch the real PAE matrix for this entry (same `entry` dict as above, no new API call)
pae_response = requests.get(entry["paeDocUrl"]).json()
pae_matrix = pae_response[0]["predicted_aligned_error"]
max_pae = pae_response[0]["max_predicted_aligned_error"]

print(f"PAE matrix shape: {len(pae_matrix)} x {len(pae_matrix[0])} residues")
print(f"Max PAE in this entry: {max_pae:.1f} Å")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(pae_matrix, cmap="Greens_r", vmin=0, vmax=max_pae)
ax.set_xlabel("Aligned residue")
ax.set_ylabel("Scored residue")
ax.set_title("Predicted Aligned Error (PAE), full-length p53")
fig.colorbar(im, ax=ax, label="Expected error (Å)")
plt.show()


### The MSA AlphaFold itself used

Both pLDDT and PAE are AlphaFold's *outputs*. One of its most important *inputs* is a
**multiple sequence alignment (MSA)**: AlphaFold does not just look at one sequence, it
looks at hundreds or thousands of evolutionarily related sequences and uses the patterns
of co-variation between positions as a major source of structural information. This is
exactly why the single-sequence `alphafold_single` mode used in the design tasks above is
a stripped-down mode, not representative AlphaFold usage — it has no MSA to draw on at
all.

AlphaFold DB actually publishes the real MSA it used for every entry (`msaUrl`, `.a3m`
format) — so instead of just asserting that MSA depth matters, we can fetch the real one
and look at it directly.

In [ ]:
# Fetch AlphaFold's own MSA for this entry and compute per-position coverage
# (how many aligned sequences have a real residue, not a gap, at each position).
msa_text = requests.get(entry["msaUrl"]).text

records = []
current = []
for line in msa_text.strip().splitlines():
    if line.startswith(">"):
        if current:
            records.append("".join(current))
        current = []
    else:
        current.append(line)
if current:
    records.append("".join(current))

query_row = records[0]
match_columns = [i for i, c in enumerate(query_row) if not c.islower()]  # a3m: uppercase/'-' = match columns

coverage = []
for col in match_columns:
    count = sum(1 for seq in records[1:] if len(seq) > col and seq[col] not in ("-", "."))
    coverage.append(count)

print(f"MSA has {len(records)} sequences (including the query itself)")
print(f"Alignment covers {len(match_columns)} match columns (== sequence length: {len(match_columns) == len(query_row.replace('-', ''))})")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(range(1, len(coverage) + 1), coverage, color="tab:purple", linewidth=1)
ax.axvspan(crystal_start, crystal_end, color="tab:green", alpha=0.15,
           label=f"3D08 resolved range ({crystal_start}-{crystal_end})")
ax.set_xlabel("Residue number (full-length p53, 1-393)")
ax.set_ylabel("# aligned sequences\nwith a residue here")
ax.set_title("MSA coverage per position")
ax.legend(loc="upper right")
plt.show()

import numpy as np
coverage_arr = np.array(coverage)
core_cov = coverage_arr[crystal_start - 1 : crystal_end]
nterm_cov = coverage_arr[: crystal_start - 1]
cterm_cov = coverage_arr[crystal_end:]
print(f"\nMean MSA coverage, core domain ({crystal_start}-{crystal_end}): {core_cov.mean():.1f} sequences/position")
print(f"Mean MSA coverage, N-terminal tail (1-{crystal_start - 1}): {nterm_cov.mean():.1f}")
print(f"Mean MSA coverage, C-terminal tail ({crystal_end + 1}-393): {cterm_cov.mean():.1f}")


**What just happened:** the *global* average for full-length p53 (~75, with ~40% of
the sequence in AlphaFold DB's low/very-low confidence bins) looks like a real weak
spot. But restricted to exactly the domain 3D08 crystallized (97-287), AlphaFold is
actually **very** confident (mean pLDDT ≈ 95-96, only ~1% below 70) — including right at
the R249 mutation hotspot itself. The pLDDT plot above shows this directly: a sharp,
visible step at the boundaries of the shaded region, not a gradual decline.

The low-confidence 40% comes almost entirely from p53's N- and C-terminal regulatory
regions, which are **intrinsically disordered** — they genuinely have no single fixed
shape in solution. AlphaFold being unconfident there isn't a prediction failure; it's the
correct answer for a region that doesn't fold into one structure.

**The MSA coverage plot adds a second, independent line of evidence for the same
pattern**, not just a restatement of it: in a live-verified run, the core domain had
roughly **2-3× deeper MSA coverage** (~100+ aligned sequences/position) than the
disordered termini (~30-50 sequences/position). ⚠️ **Read this as a correlation, stated
honestly, not a proof of causation**: intrinsically disordered regions are *both* harder
to align confidently (lower coverage — evolution doesn't conserve a sequence that doesn't
need a fixed fold) *and* independently lack a single native structure (the reason already
given above). Don't claim low MSA coverage *causes* low pLDDT here — the two are
consistent with each other, pointing at the same underlying biology, not one explaining
the other.

This is exactly the kind of thing to check before trusting (or dismissing) an AlphaFold
confidence score: a single overall number can hide very different stories in different
parts of the same protein, and now you have three independent views of that same story —
pLDDT, PAE, and the real MSA — not just one.

---

## 🔭 FRONTIER → Block C (Molecular Dynamics)

**⏭ Signpost:** You'll fully understand *why* this happens in Lectures 7–9, when we
cover molecular dynamics.

**The question:** AlphaFold gave p53's core domain (97-287) a very high, uniform
confidence score (~95-96 mean pLDDT) — but a pLDDT score is a **static, single-snapshot**
measure. Even within this "confident" domain, do you predict every residue will be
equally *rigid* in an actual molecular dynamics simulation, or do you expect some region
to move noticeably more than others despite the uniformly high pLDDT?

**💡 Hint:** pLDDT measures how confident AlphaFold is about *where* a residue sits in
one predicted structure — it is not, by itself, a direct measurement of flexibility in
solution. A region can be confidently placed and still move.

**🎁 Low-stakes:** Bonus/calibration item — does not block finishing this exercise;
being wrong here is expected and not penalized.

**✅ Minimum viable answer:** "I predict [region] will move the most because ___." A
one-line, defended guess is complete.

*Your answer:*


---

## Correlating the AlphaFold model with the 3D08 crystal: what actually changed?

Confidence metrics are AlphaFold's own self-assessment. The more direct check is to put
the AlphaFold model and the real crystal structure **side by side** and measure the
difference directly.

Two things worth knowing *before* looking at the numbers, so a deviation isn't
misread:

- AlphaFold DB's model is predicted from p53's **canonical (wild-type) UniProt
  sequence** (`entry["sequence"]`, already fetched above). 3D08's crystal carries the
  **R249S mutation** — so this comparison is partly "AlphaFold's WT prediction vs. a
  mutant crystal," not purely "prediction accuracy vs. experiment." A deviation right at
  residue 249 could reflect either.
- 3D08's crystal has a bound **structural Zn²⁺ ion**, coordinated by four residues
  (verified below). AlphaFold's model is **protein-only** — it does not predict ions,
  ligands, or cofactors at all. That's a categorical omission by design, not something an
  RMSD number will show you.

In [ ]:
# Download both structures: AlphaFold's model (already have its URL in `entry`)
# and the 3D08 crystal (RCSB).
af_model_text = requests.get(entry["pdbUrl"]).text
crystal_text = requests.get("https://files.rcsb.org/download/3D08.pdb").text


def parse_ca_atoms(pdb_text, chain="A"):
    """Parse CA coordinates from PDB ATOM records, keyed by residue number.

    Uses fixed-width column parsing (the actual PDB format), not whitespace
    splitting -- whitespace splitting silently misparses some lines (e.g. when
    columns run together at 4-digit residue numbers).
    """
    coords = {}
    for line in pdb_text.splitlines():
        if not line.startswith("ATOM"):
            continue
        if line[12:16].strip() != "CA":
            continue
        if line[16] not in (" ", "A"):  # skip alternate conformations except the first
            continue
        if line[21] != chain:
            continue
        resnum = int(line[22:26])
        xyz = (float(line[30:38]), float(line[38:46]), float(line[46:54]))
        coords[resnum] = xyz
    return coords


af_ca = parse_ca_atoms(af_model_text)
crystal_ca = parse_ca_atoms(crystal_text)
print(f"AlphaFold model: {len(af_ca)} CA atoms, range {min(af_ca)}-{max(af_ca)}")
print(f"3D08 crystal: {len(crystal_ca)} CA atoms, range {min(crystal_ca)}-{max(crystal_ca)}")

# 3D08's crystal is NOT a clean, gap-free span -- match by residue number, not by
# list position, or a naive pairing would silently misalign everything past a gap.
common_residues = sorted(set(af_ca) & set(crystal_ca))
missing_from_crystal = [r for r in range(min(common_residues), max(common_residues) + 1)
                         if r not in crystal_ca]
print(f"Matched (by residue number) residues: {len(common_residues)}")
print(f"Residues with no crystal coordinates inside that range (unresolved loops): {missing_from_crystal}")


In [ ]:
import numpy as np

# Superimpose the AlphaFold model onto the crystal using the matched CA atoms
# (Kabsch algorithm) -- this is the actual math behind "structural alignment."
P = np.array([af_ca[r] for r in common_residues])       # AlphaFold model (mobile)
Q = np.array([crystal_ca[r] for r in common_residues])  # crystal (reference)

P_centroid, Q_centroid = P.mean(axis=0), Q.mean(axis=0)
Pc, Qc = P - P_centroid, Q - Q_centroid

H = Pc.T @ Qc
U, S, Vt = np.linalg.svd(H)
d = np.sign(np.linalg.det(Vt.T @ U.T))
correction = np.diag([1, 1, d])  # guards against an improper rotation (reflection)
R = Vt.T @ correction @ U.T

P_aligned = (R @ Pc.T).T + Q_centroid
per_residue_deviation = np.linalg.norm(P_aligned - Q, axis=1)
overall_rmsd = np.sqrt((per_residue_deviation ** 2).mean())

print(f"Overall CA RMSD over {len(common_residues)} matched residues: {overall_rmsd:.2f} Å")

deviation_by_residue = dict(zip(common_residues, per_residue_deviation))
worst = sorted(deviation_by_residue.items(), key=lambda kv: -kv[1])[:10]
print("\nHighest-deviation residues:")
for resnum, dev in worst:
    print(f"  residue {resnum}: {dev:.2f} Å")

print(f"\nDeviation right at the R249S hotspot: {deviation_by_residue.get(249, float('nan')):.2f} Å")


**What this shows, in a live-verified run:** an overall CA RMSD around 0.7-0.8 Å across
the whole matched domain — AlphaFold's fold-level prediction of the core domain is very
close to the crystal, wild-type-vs-mutant notwithstanding. Deviation right at residue
249 itself is small, similar in size to the average — the R249S mutation changes a side
chain, not the backbone path there. The **highest**-deviation residues instead cluster
immediately **next to the crystal's own unresolved loops** (missing residues printed
above) — which makes sense: a static AlphaFold prediction is compared against one
specific crystal snapshot, and the regions right at the edge of a genuinely
flexible/disordered loop are exactly where a single crystal conformation and a single
predicted conformation are least likely to agree, independent of whether the prediction
is "wrong." ⚠️ Exact numbers depend on the live AlphaFold DB model version — re-verify if
this section is ever re-run and the numbers look different.

In [ ]:
# Locate the residues that actually coordinate the structural Zn (within 2.6 Å,
# a typical direct-coordination distance) -- confirms the claim above with real
# geometry, and gives us something concrete to zoom in on.
zn_position = None
for line in crystal_text.splitlines():
    if line.startswith("HETATM") and line[17:20].strip() == "ZN":
        zn_position = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
        break

zn_coordinating_residues = []
for line in crystal_text.splitlines():
    if not line.startswith("ATOM"):
        continue
    resnum = int(line[22:26])
    xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    if np.linalg.norm(xyz - zn_position) < 2.6:
        zn_coordinating_residues.append(resnum)

zn_coordinating_residues = sorted(set(zn_coordinating_residues))
print(f"Residues directly coordinating the structural Zn²⁺: {zn_coordinating_residues}")
print("(A classic Cys/His zinc-finger-like site -- present in the crystal, absent from the AlphaFold model.)")


In [ ]:
# Apply the same rotation+translation to the FULL AlphaFold model (not just the
# matched CA atoms), so the whole structure moves together, then overlay both
# structures in one py3Dmol view.
def apply_transform(pdb_text, R, translation_from, translation_to):
    new_lines = []
    for line in pdb_text.splitlines():
        if line.startswith(("ATOM", "HETATM")):
            xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            new_xyz = (R @ (xyz - translation_from)) + translation_to
            line = f"{line[:30]}{new_xyz[0]:8.3f}{new_xyz[1]:8.3f}{new_xyz[2]:8.3f}{line[54:]}"
        new_lines.append(line)
    return "\n".join(new_lines)


af_model_aligned_text = apply_transform(af_model_text, R, P_centroid, Q_centroid)

view = py3Dmol.view(width=900, height=550)
view.addModel(crystal_text, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "skyblue"}})
view.addModel(af_model_aligned_text, "pdb")
view.setStyle({"model": 1}, {"cartoon": {"color": "orange"}})

# Zoom on the R249S hotspot and the Zn-coordinating residues together
highlight_resi = [249] + zn_coordinating_residues
view.addStyle({"model": 0, "resi": highlight_resi}, {"stick": {"color": "skyblue"}})
view.addStyle({"model": 1, "resi": highlight_resi}, {"stick": {"color": "orange"}})
view.addStyle({"model": 0, "resn": "ZN"}, {"sphere": {"color": "grey", "radius": 0.6}})

view.zoomTo({"model": 0, "resi": highlight_resi})
view.show()
print("Blue = 3D08 crystal (R249S mutant, with bound Zn). Orange = AlphaFold's WT model (no Zn, no ligand).")


**Your turn:** repeat this analysis for the protein you chose in ex01's Character Sheet,
**before** starting Task P.1 below. You now have a worked template for all of it, not
just the confidence lookup:
- global mean pLDDT, confidence-bin fractions, and a zoom-in on your structure's own
  resolved range (as before);
- a per-residue **pLDDT plot** and a **PAE heatmap**, both straight from AlphaFold DB;
- an **MSA coverage plot** using AlphaFold's own real MSA for your protein (⚠️ not every
  protein will have as deep an MSA as p53's 1,455 sequences — a well-studied human gene
  is not typical, don't expect that number as a baseline);
- a **structural comparison against a crystal structure** (RMSD, per-residue deviation,
  overlay) if your protein has both an AlphaFold DB entry and a solved experimental
  structure — if it only has one or the other, say so explicitly rather than forcing the
  comparison.

---

# Project Assignment: Protein Disruption Analysis  

## Background

Now apply what you learned to YOUR project protein. The goal is to:
1. Predict your protein's structure
2. **Rationally design** mutations that disrupt it
3. **Test hypotheses** about which regions are critical

## Part 1: Baseline Prediction  

### Task P.1: AlphaFold Prediction vs Crystal Structure

**First, check [AlphaFold Database](https://alphafold.ebi.ac.uk/)** - your protein might already be there!

If not, use:
- [AlphaFold Server](https://alphafoldserver.com/welcome),
- [AlphaFold Colab](https://colab.research.google.com/github/deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb),
- OR [ColabFold](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb) (faster)

**Required analysis** — you now have a worked template above for every one of these,
built on p53/3D08. Use the same tools (AlphaFold DB API for pLDDT/PAE/MSA, the
Kabsch-superposition cell for RMSD), not new ones.

**Questions to answer:**
1. **What is the mean pLDDT score?**
   - Overall:
   - Interpretation:

2. **Which regions have highest confidence (pLDDT > 90)?**
   - Residues:
   - Why? (secondary structure, core vs surface):

3. **Which regions have lowest confidence (pLDDT < 70)?**
   - Residues:
   - Why? (loops, disordered, termini):

4. **RMSD with crystal structure:**
   - Overall RMSD:
   - Method used (e.g. the Kabsch-superposition cell above, or PyMOL `align`/`rms_cur`):
   - Regions of high deviation:
   - Why these regions differ (check: near an unresolved crystal loop? a mutation? a
     genuinely wrong prediction?):

5. **PAE (Predicted Aligned Error) analysis:**
   - Well-resolved domains:
   - Uncertain inter-domain contacts:
   - What this tells you about protein architecture: 

## Part 2: Rational Mutation Design  

### Task P.2: Hypothesis-Driven Disruption

**BEFORE testing mutations**, complete this analysis:

#### Strategy 1: Core Disruption
**Target:** Hydrophobic core residues

**Identify core residues:**
- Which residues form the hydrophobic core? (use structure viewer)
- List at least 3 candidates: 

**Design mutations:**
- Mutation 1: [original]→[mutant] at position [X]
  - Why this will disrupt: 
  - Prediction: pLDDT will drop to ~[value]

- Mutation 2: 
  - Reasoning: 

- Mutation 3: 
  - Reasoning: 

#### Strategy 2: Helix/Sheet Disruption  
**Target:** Critical secondary structure elements

**Identify critical secondary structures:**
- Which helix/sheet is most important? Why?
- Residues: 

**Design mutations:**
- Proline insertion at position [X] because...
- Expected effect: 

#### Strategy 3: Salt Bridge Disruption
**Target:** Electrostatic interactions

**Identify salt bridges:**
- [Residue A] with [Residue B]
- Distance: 
- Importance: 

**Design mutations:**
- Mutation: 
- Why disruptive: 

#### Strategy 4: Disulfide Bond Disruption (if applicable)
**If your protein has disulfide bonds:**
- Which cysteines form bonds?
- Mutation strategy: 

## Part 3: Testing and Analysis  

### Task P.3: Predict Mutants

Test your designed mutations. For EACH mutation:

**Mutation [name]:**
- Hypothesis: 
- Predicted pLDDT: 
- Actual pLDDT: 
- Structure changes observed: 
- Did hypothesis hold?: 
- Explanation: 

### Summary Analysis

**Most disruptive mutation type:**

**Least disruptive:**

**Surprising findings:**

**What this reveals about your protein's stability:**

**Clinical/functional implications:**
- If these mutations occurred naturally: 
- Disease relevance: 

## Reference: Protein Stabilizing Forces

<img src="https://github.com/yerkoescalona/structural_bioinformatics/blob/w2023/ex02/structure_stabilizing_forces.png?raw=1" width="600"/>

## Bonus: MSA Analysis  

Use [Clustal Omega](https://toolkit.tuebingen.mpg.de/tools/clustalo) or this [MSA notebook](https://colab.research.google.com/github/yerkoescalona/structural_bioinformatics/blob/main/ex02/msa.ipynb)

**Question:** Are the residues you targeted for mutation conserved across species?
- If yes: Strong evidence they're critical
- If no: Why might non-conserved residues still be important in your specific protein?






Happy modeling!